# **Notebook 7: Solution V2 — Fine-Tuned RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./intent_lora_best/` — Created by Notebook 6
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` + `v1_metrics.csv` — From Notebooks 3/4/5
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `Comparative_Results_Full.csv` + `Comparative_Results_Summary.csv` _(Final deliverables)_

---

### **Task 4.3: Integrate Fine-Tuned Model with Retrieval**

#### **4.3.1 Integrate Fine-Tuned Model into Existing RAG Pipeline [3 marks]**
**The Task:** Replace the baseline model with the fine-tuned model acting as an intent router. Merge the LoRA adapters, extract a JSON intent, map it to a vector-search string, retrieve, and generate. Validate the integrated system.

**Hints & Tips:**
* `PeftModel.from_pretrained(base_model, "./intent_lora_best").merge_and_unload()` fuses the adapters for fast inference.
* Use a strong system prompt with few-shot examples so the router emits JSON only; `re.search(r'\{.*?\}', raw)` is a safety net for stray preamble.
* Map the intent (e.g. `track_order`) to an SOP header search string (e.g. `# Track Order`). Fall back to the raw query if JSON parsing fails.
* Validate end-to-end on `test_query`: intent → search string → retrieved SOP → final answer.

**Parameter Tuning:**
* `max_new_tokens=30` for the router (JSON is short — more tokens invite trailing explanation text).
* 4 few-shot examples is the sweet spot.

**Learner Inference:** Querying with the structured intent keyword instead of the noisy prompt retrieves the exact policy clause — the core of Hybrid RAG.

In [1]:
!pip install -U ragas langchain langchain-core langchain-community langchain-huggingface langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 9.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.5 MB/s eta 0:00:00:00:0100:01


In [3]:
# YOUR CODE HERE
import os
import re
import json
import torch
import shutil
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# ── Config ───────────────────────────────────────────────
MODEL_ID        = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_BEST_PATH  = "/kaggle/input/datasets/piyushshx/artifacts-2/intent_lora_best"
CHROMA_PATH     = "/kaggle/working/chroma_db"
TEST_CSV        = "/kaggle/input/datasets/piyushshx/artifacts/df_test.csv"
OUTPUTS_JSON    = "/kaggle/input/datasets/piyushshx/artifacts-2/outputs.json"
V1_METRICS_CSV  = "/kaggle/input/datasets/piyushshx/artifacts-2/v1_metrics.csv"

os.makedirs("/kaggle/working/artifacts", exist_ok=True)

# Copy unzipped chroma_db directly to working
src = "/kaggle/input/datasets/piyushshx/artifacts-2/chroma_db/chroma_db"
dst = "/kaggle/working/chroma_db"

if os.path.exists(dst):
    shutil.rmtree(dst)

shutil.copytree(src, dst)
print("ChromaDB copied ✅")
print(os.listdir(dst))
# ── Load Base Model in 4-bit ─────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
tokenizer  = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Base model loaded ✅")

# ── Load & Merge LoRA Adapters ───────────────────────────
model = PeftModel.from_pretrained(base_model, LORA_BEST_PATH)
model = model.merge_and_unload()
model.eval()
print("LoRA adapters merged ✅")

# ── Load ChromaDB ────────────────────────────────────────
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectordb = Chroma(
    persist_directory="/kaggle/working/chroma_db",
    embedding_function=embedding_function
)
print(f"ChromaDB loaded — {vectordb._collection.count()} docs ✅")

# ── Load Test Data ───────────────────────────────────────
test_df = pd.read_csv(TEST_CSV)
with open(OUTPUTS_JSON) as f:
    outputs = json.load(f)
v1_metrics = pd.read_csv(V1_METRICS_CSV)
print(f"Test set: {len(test_df)} rows ✅")

ChromaDB copied ✅
['chroma.sqlite3', '9a2196dc-77be-4e35-883d-9bc503881273']


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB loaded — 13 docs ✅
Test set: 391 rows ✅


In [4]:
import re
import json
import torch

def get_sop_reference(query):
    docs = vectordb.similarity_search(query, k=1)
    return docs[0].page_content if docs else ""

def parse_response(response):
    cleaned = re.sub(r"```json|```", "", response).strip()
    try:
        return json.loads(cleaned), True
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', cleaned, re.DOTALL)   # fallback for stray preamble
        if match:
            try:
                return json.loads(match.group()), True
            except json.JSONDecodeError:
                pass
        return {}, False

def intent_to_search_string(intent):
    return intent.replace("_", " ").title()

def run_hybrid_rag(query, k=1):
    # Step 1: extract intent via the fine-tuned router
    messages = [
        {"role": "system", "content": "You are a customer support intent classifier. Extract the intent and category from the customer query and return a valid JSON object only. No explanation."},
        {"role": "user", "content": query}
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=30, do_sample=False, temperature=None, top_p=None)

    raw_response = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    parsed, is_valid = parse_response(raw_response)

    # Step 2: map intent to a search string (fall back to the raw query)
    search_string = intent_to_search_string(parsed["intent"]) if is_valid and "intent" in parsed else query

    # Step 3: retrieve the SOP with the structured search string
    retrieved_docs = vectordb.similarity_search(search_string, k=k)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    sources = [doc.metadata['filename'] for doc in retrieved_docs]

    # Step 4: generate the final response with SOP context
    final_messages = [
        {"role": "system", "content": f"Answer strictly using this SOP: {context}\n\nReturn a valid JSON object with intent and category only. No explanation."},
        {"role": "user",   "content": query}
    ]
    final_text   = tokenizer.apply_chat_template(final_messages, tokenize=False, add_generation_prompt=True)
    final_inputs = tokenizer(final_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        final_ids = model.generate(**final_inputs, max_new_tokens=120, do_sample=False, temperature=None, top_p=None)

    final_response = tokenizer.decode(final_ids[0][final_inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    final_parsed, final_valid = parse_response(final_response)

    return {
        "router_output":     raw_response,
        "parsed_intent":     parsed.get("intent", ""),
        "search_string":     search_string,
        "retrieved_sources": sources,
        "final_response":    final_response,
        "router_valid_json": is_valid,      # router step only
        "valid_json":        final_valid,   # final response, the one that gets scored
    }

# End-to-end validation: intent -> search string -> retrieved SOP -> final answer
print("=== End-to-End Validation ===\n")
for q in ["i ordered something 2 weeks ago and it still hasn't arrived!",
          "i need a refund for my order",
          "how do i reset my password?"]:
    r = run_hybrid_rag(q)
    print(f"Query:          {q}")
    print(f"Router output:  {r['router_output']}")
    print(f"Search string:  {r['search_string']}")
    print(f"Retrieved SOP:  {r['retrieved_sources']}")
    print(f"Final response: {r['final_response']}")
    print("-" * 60)

=== End-to-End Validation ===

Query:          i ordered something 2 weeks ago and it still hasn't arrived!
Router output:  {"intent": "track_order", "category": "order_status"}
Search string:  Track Order
Retrieved SOP:  ['order_tracking.md']
Final response: {"intent": "delivery_status", "category": "order_tracking"}
------------------------------------------------------------
Query:          i need a refund for my order
Router output:  {"intent": "refund", "category": "payment"}
Search string:  Refund
Retrieved SOP:  ['refund_policy.md']
Final response: {"intent": "refund", "category": "refunding"}
------------------------------------------------------------
Query:          how do i reset my password?
Router output:  {"intent": "set_password", "category": "account"}
Search string:  Set Password
Retrieved SOP:  ['password_reset.md']
Final response: click here to reset your password: [link]
------------------------------------------------------------


### **Task 4.4: Evaluate Solution V2**

#### **4.4.1 Re-Execute Evaluation Framework [3 marks]**
**The Task:** Evaluate Format Adherence and Intent Accuracy on the held-out test split (zero leakage guaranteed) and an adversarial subset derived via regex filtering. Evaluate the final synthesis using ROUGE/BLEU.

**Hints & Tips:**
* Reuse `df_test` from Notebook 2 — it's the leakage-free test split.
* Build the adversarial subset by regex-filtering for sentiment/hedging words (`still`, `never`, `terrible`, `frustrated`).
* Report Format Adherence %, Exact Match %, and Fuzzy Match % (fuzzy catches `order_tracking` vs `track_order`).

**Learner Inference:** Using the held-out test split guarantees zero leakage and trustworthy scores.

In [5]:
!pip install -q rouge-score nltk
import re
import nltk
nltk.download('punkt', quiet=True)

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tqdm import tqdm

# ── Adversarial keywords (your list from the word-count step) ──
adversarial_keywords = ['again', 'problem', 'issue', 'wrong', 'help','cannot', "can't"] 
pattern = r"\b(?:" + "|".join(re.escape(w) for w in adversarial_keywords) + r")\b"

# ── Helpers ──────────────────────────────────────────────
rouge  = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

def get_sop_reference(query):
    docs = vectordb.similarity_search(query, k=1)
    return docs[0].page_content if docs else ""

def evaluate_v2(eval_df):
    results = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Held-out test split"):
        query        = row['instruction']
        true_intent  = row['intent'].lower().strip()

        result           = run_hybrid_rag(query, k=1)
        predicted_intent = result['parsed_intent'].lower().strip()
        final_response   = result['final_response']

        exact_match = predicted_intent == true_intent

        pred_words  = set(predicted_intent.replace("_", " ").split())
        true_words  = set(true_intent.replace("_", " ").split())
        fuzzy_match = len(pred_words & true_words) / max(len(true_words), 1) >= 0.5

        reference = get_sop_reference(query)
        r_scores  = rouge.score(reference, final_response)
        bleu      = sentence_bleu(
            [reference.split()], final_response.split(),
            smoothing_function=smooth
        )

        results.append({
            "query":            query,
            "true_intent":      true_intent,
            "predicted_intent": predicted_intent,
            "valid_json":       result['valid_json'],
            "exact_match":      exact_match,
            "fuzzy_match":      fuzzy_match,
            "rouge1":           r_scores['rouge1'].fmeasure,
            "rougeL":           r_scores['rougeL'].fmeasure,
            "bleu":             bleu,
            "final_response":   final_response,
            "router_valid_json": result['router_valid_json'],
        })
    return pd.DataFrame(results)

def report(df_results, label):
    total = len(df_results)
    if total == 0:
        print(f"\n=== {label}: no rows, widen the keywords ===")
        return
    print(f"\n=== {label} Results ===")
    print(f"Format Adherence: {df_results['valid_json'].sum()}/{total} ({df_results['valid_json'].mean()*100:.1f}%)")
    print(f"Router Format Adherence: {df_results['router_valid_json'].mean()*100:.1f}%")
    print(f"Exact Match:      {df_results['exact_match'].sum()}/{total} ({df_results['exact_match'].mean()*100:.1f}%)")
    print(f"Fuzzy Match:      {df_results['fuzzy_match'].sum()}/{total} ({df_results['fuzzy_match'].mean()*100:.1f}%)")
    print(f"ROUGE-1:          {df_results['rouge1'].mean():.3f}")
    print(f"ROUGE-L:          {df_results['rougeL'].mean():.3f}")
    print(f"BLEU:             {df_results['bleu'].mean():.3f}")

# ── Full held-out test split (one run) ───────────────────
v2_results = evaluate_v2(test_df)
report(v2_results, "Full Test Set")

# ── Adversarial subset: regex-filtered rows of the same run ──
adversarial_df = test_df[test_df['instruction'].str.lower().str.contains(pattern, regex=True)]
print(f"\nFull test set: {len(test_df)} rows | Adversarial subset: {len(adversarial_df)} rows")
v2_adv_results = v2_results[v2_results['query'].isin(adversarial_df['instruction'])]
report(v2_adv_results, "Adversarial Subset")

Held-out test split: 100%|██████████| 391/391 [16:59<00:00,  2.61s/it]


=== Full Test Set Results ===
Format Adherence: 364/391 (93.1%)
Router Format Adherence: 98.2%
Exact Match:      133/391 (34.0%)
Fuzzy Match:      264/391 (67.5%)
ROUGE-1:          0.021
ROUGE-L:          0.018
BLEU:             0.000

Full test set: 391 rows | Adversarial subset: 101 rows

=== Adversarial Subset Results ===
Format Adherence: 96/101 (95.0%)
Router Format Adherence: 98.0%
Exact Match:      34/101 (33.7%)
Fuzzy Match:      70/101 (69.3%)
ROUGE-1:          0.019
ROUGE-L:          0.017
BLEU:             0.000


#### **4.4.2 Analyse Fine-Tuning Impact [2 marks]**
**The Task:** Compare Solution V1 (Naive RAG) against Solution V2 (Hybrid RAG) to quantify the improvement attributable to fine-tuning.

**Hints & Tips:**
* Load `v1_metrics.csv` from Notebook 5 and compare against the V2 scores you just computed.
* Compute improvement percentages: `(v2 - v1) / v1 * 100` for each metric.
* Attribute the delta specifically to fine-tuning — retrieval was already present in V1, so any gain here is the router's contribution.

**Learner Inference:** This isolates fine-tuning's contribution, just as Task 3.4 isolated retrieval's — together they decompose the full system's improvement.

In [6]:
import numpy as np

# V1 (Naive RAG) scored on the same rows as Task 4.4.1, from the saved NB4 outputs
saved   = {r['test_query']: r for r in outputs}
v1_rows = []
for _, row in test_df.iterrows():
    q, true_intent = row['instruction'], row['intent'].lower().strip()
    raw = saved[q]['naive_rag_output']
    parsed, valid = parse_response(raw)
    ref = get_sop_reference(q)
    r   = rouge.score(ref, raw)
    v1_rows.append({
        "valid_json":  valid,
        "exact_match": str(parsed.get("intent", "")).lower().strip() == true_intent,
        "rouge1":      r['rouge1'].fmeasure,
        "rougeL":      r['rougeL'].fmeasure,
        "bleu":        sentence_bleu([ref.split()], raw.split(), smoothing_function=smooth),
    })
v1_res = pd.DataFrame(v1_rows)
assert len(v1_res) == len(v2_results), "V1 and V2 must be scored on the same rows"

comparison = pd.DataFrame({
    "metric": ["Format Adherence %", "Intent Accuracy % (exact)", "ROUGE-1", "ROUGE-L", "BLEU"],
    "v1_naive_rag": [v1_res["valid_json"].mean()*100, v1_res["exact_match"].mean()*100,
                     v1_res["rouge1"].mean(), v1_res["rougeL"].mean(), v1_res["bleu"].mean()],
    "v2_hybrid":    [v2_results["valid_json"].mean()*100, v2_results["exact_match"].mean()*100,
                     v2_results["rouge1"].mean(), v2_results["rougeL"].mean(), v2_results["bleu"].mean()],
})
comparison["delta"] = (comparison["v2_hybrid"] - comparison["v1_naive_rag"]).round(3)
comparison["improvement_%"] = np.where(comparison["v1_naive_rag"] == 0, np.nan,
                                       (comparison["delta"] / comparison["v1_naive_rag"] * 100).round(1))

print(f"=== V1 (Naive RAG) vs V2 (Hybrid RAG), {len(v2_results)} test rows ===")
print(comparison.round(3).to_string(index=False))
print(f"\nFuzzy Match (V2 only): {v2_results['fuzzy_match'].mean()*100:.1f}%")

=== V1 (Naive RAG) vs V2 (Hybrid RAG), 391 test rows ===
                   metric  v1_naive_rag  v2_hybrid  delta  improvement_%
       Format Adherence %        94.629     93.095 -1.535           -1.6
Intent Accuracy % (exact)         1.023     34.015 32.992         3225.0
                  ROUGE-1         0.026      0.021 -0.004          -15.5
                  ROUGE-L         0.023      0.018 -0.004          -17.7
                     BLEU         0.000      0.000 -0.000           -0.0

Fuzzy Match (V2 only): 67.5%


### **Task 4.5: Perform Comparative Analysis**

> Subtasks 4.5.1 (Compare All Versions) and 4.5.2 (Document Findings) are written up in the **Comparative Analysis Report PDF**. The cell below generates the scoring tables that feed that report.

**The Task:** Run all three architectures (Baseline, Naive RAG, Hybrid RAG) across the full held-out test split with SOP-grounded references, then export the per-row and summary CSVs.

**Hints & Tips:**
* SOP-grounded references reward policy-specific answers, ensuring Hybrid scores highest.
* This is the most compute-intensive cell — expect 15–30 min on T4. Use `df_test.head(50)` if time-constrained.
* Export `Comparative_Results_Full.csv` (per-row) and `Comparative_Results_Summary.csv` (aggregate).

In [7]:
from tqdm import tqdm

saved   = {r['test_query']: r for r in outputs}
eval_df = test_df                                   # full held-out test split
print("Queries missing from outputs.json:", sum(q not in saved for q in eval_df['instruction']))
print("naive_rag_output present:", 'naive_rag_output' in outputs[0])

def fuzzy_match(pred, true):
    p = set(pred.replace("_", " ").split())
    t = set(true.replace("_", " ").split())
    return len(p & t) / max(len(t), 1) >= 0.5

def check_hallucination(response, sop_context):
    patterns = [
        r'\d+\s*(hours|days|weeks|business days)',
        r'department\s+\w+',
        r'call\s+us\s+at',
        r'tracking\s+number\s+\w+',
    ]
    resp, sop = response.lower(), sop_context.lower()
    return any(re.search(p, resp) and not re.search(p, sop) for p in patterns)

def score_row(system, query, response, pred_intent, true_intent, valid, reference):
    r = rouge.score(reference, response)
    return {
        "system":           system,
        "query":            query,
        "true_intent":      true_intent,
        "predicted_intent": pred_intent,
        "valid_json":       valid,
        "exact_match":      pred_intent == true_intent,
        "fuzzy_match":      fuzzy_match(pred_intent, true_intent),
        "rouge1":           r['rouge1'].fmeasure,
        "rougeL":           r['rougeL'].fmeasure,
        "bleu":             sentence_bleu([reference.split()], response.split(), smoothing_function=smooth),
        "hallucination":    check_hallucination(response, reference),
    }

# Hybrid RAG outputs come from the Task 4.4.1 run (same model, greedy decoding)
hybrid = {r['query']: r for r in v2_results.to_dict('records')}

rows = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    q           = row['instruction']
    true_intent = row['intent'].lower().strip()
    reference   = get_sop_reference(q)

    base_raw = saved[q]['baseline_output']
    bp, bv   = parse_response(base_raw)
    rows.append(score_row("Baseline", q, base_raw, str(bp.get("intent", "")).lower().strip(), true_intent, bv, reference))

    rag_raw = saved[q]['naive_rag_output']
    rp, rv  = parse_response(rag_raw)
    rows.append(score_row("Naive RAG", q, rag_raw, str(rp.get("intent", "")).lower().strip(), true_intent, rv, reference))

    h = hybrid[q]
    rows.append(score_row("Hybrid RAG", q, h['final_response'], h['predicted_intent'], true_intent, bool(h['valid_json']), reference))

full_df = pd.DataFrame(rows)
full_df.to_csv("/kaggle/working/Comparative_Results_Full.csv", index=False)
print(f"Comparative_Results_Full.csv saved ✅ ({len(full_df)} rows)")

Queries missing from outputs.json: 0
naive_rag_output present: True


100%|██████████| 391/391 [00:07<00:00, 55.60it/s]

Comparative_Results_Full.csv saved ✅ (1173 rows)


In [8]:
summary = full_df.groupby("system").agg(
    format_adherence_pct = ("valid_json",    lambda s: s.mean() * 100),
    exact_match_pct      = ("exact_match",   lambda s: s.mean() * 100),
    fuzzy_match_pct      = ("fuzzy_match",   lambda s: s.mean() * 100),
    rouge1               = ("rouge1",        "mean"),
    rougeL               = ("rougeL",        "mean"),
    bleu                 = ("bleu",          "mean"),
    hallucination_pct    = ("hallucination", lambda s: s.mean() * 100),
).reindex(["Baseline", "Naive RAG", "Hybrid RAG"]).round(3)

# Consistency: Baseline / Naive RAG read from the NB5 file, Hybrid RAG measured here
vm = v1_metrics.copy()
vm["key"] = vm["metric"].str.replace("%", "", regex=False).str.strip().str.lower()
vm = vm.set_index("key")

consistency_v2 = 100 * sum(
    len({run_hybrid_rag(q)["final_response"] for _ in range(3)}) == 1
    for q in eval_df["instruction"].head(5)
) / 5

summary["consistency_pct"] = [
    vm.loc["consistency", "baseline"],
    vm.loc["consistency", "naive_rag"],
    consistency_v2,
]

summary.to_csv("/kaggle/working/Comparative_Results_Summary.csv")
print(summary.T.to_string())
print("\nComparative_Results_Summary.csv saved ✅")

system                Baseline  Naive RAG  Hybrid RAG
format_adherence_pct   100.000     94.629      93.095
exact_match_pct          1.790      1.023      34.015
fuzzy_match_pct          7.673     19.949      67.519
rouge1                   0.014      0.026       0.021
rougeL                   0.014      0.023       0.018
bleu                     0.000      0.000       0.000
hallucination_pct        0.000      0.000       1.023
consistency_pct        100.000    100.000     100.000

Comparative_Results_Summary.csv saved ✅


---
## END-OF-NOTEBOOK CHECKLIST (FINAL)

> **IMPORTANT: This is the last graded notebook. Verify all deliverables.**

- [ ] **4.3.1** LoRA merged + Hybrid RAG integration validated (intent → search → retrieve → generate)
- [ ] **4.4.1** Format Adherence + Exact Match + Fuzzy Match on test split + adversarial subset
- [ ] **4.4.2** Fine-tuning impact quantified (V1 vs V2 with %)
- [ ] **4.5** All 3 architectures scored with SOP-grounded references
- [ ] **`Comparative_Results_Full.csv` saved** ← _FINAL DELIVERABLE_
- [ ] **`Comparative_Results_Summary.csv` saved** ← _FINAL DELIVERABLE_

### Complete Artifact Inventory

| Artifact | Created In |
|---|---|
| `sampled_data.csv` | NB1 |
| `./tokenized_train/`, `./tokenized_valid/`, `df_test.csv` | NB2 |
| `outputs.json` | NB3 + NB4 |
| `./chroma_db/` | NB4 |
| `v1_metrics.csv` | NB5 |
| `./intent_lora_best/`, `training_log.csv`, `training_curves.png` | NB6 |
| `Comparative_Results_Full.csv`, `Comparative_Results_Summary.csv` | NB7 |

**Mark all items checked, then prepare your final submission package.**